In [ ]:
# !pip uninstall -y torch torchvision torchaudio

# !pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 \
# --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# !pip install "numpy<2" --force-reinstall

In [ ]:
import numpy as np
import torch

print("numpy:", np.__version__)
print("torch:", torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_arch_list())

In [ ]:
!pip install -q evaluate rouge-score sentencepiece safetensors

In [ ]:
import os, shutil

# Remove the corrupted download
if os.path.exists("/kaggle/working/mt5-small-model"):
    shutil.rmtree("/kaggle/working/mt5-small-model")

# Re-download with HF_TOKEN disabled (avoids wrong model)
from huggingface_hub import snapshot_download
import os
os.environ.pop("HF_TOKEN", None)  # remove any token
snapshot_download(repo_id="google/mt5-small", local_dir="/kaggle/working/mt5-small-model")

In [ ]:
# Verify it's mT5, not Mistral
import json
with open("/kaggle/working/mt5-small-model/config.json") as f:
    config = json.load(f)
print(config.get("model_type"))
# Should print: t5

In [ ]:
import os
import gc
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)

import evaluate

In [ ]:
MODEL_NAME = "google/mt5-small"

DATA_PATH = "/kaggle/input/datasets/janenjuguna/multilingual-health-data"

TRAIN_PATH = os.path.join(DATA_PATH, "Train.csv")
VAL_PATH = os.path.join(DATA_PATH, "Val.csv")
TEST_PATH = os.path.join(DATA_PATH, "Test.csv")
SAMPLE_PATH = os.path.join(DATA_PATH, "SampleSubmission.csv")

OUTPUT_DIR = "./mt5-health-model"
BEST_DIR = "./mt5-health-model/best_model"

MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 160

BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 3e-5
NUM_EPOCHS = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_df = pd.read_csv(SAMPLE_PATH)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)
print("Sample:", sample_df.shape)

print(train_df.columns.tolist())
print(val_df.columns.tolist())
print(test_df.columns.tolist())

display(train_df.head())
display(val_df.head())
display(test_df.head())

In [ ]:
# Your actual columns are:
# Train/Val: ID, input, output, subset
# Test: ID, input, subset
# input = real question
# output = answer

def clean_train_val(df):
    df = df.copy()

    df = df.dropna(subset=["input", "output", "subset"])

    df["input"] = df["input"].astype(str).str.strip()
    df["output"] = df["output"].astype(str).str.strip()
    df["subset"] = df["subset"].astype(str).str.strip()

    df = df[(df["input"] != "") & (df["output"] != "")]

    # Add language-country prefix to help the model know the target language
    df["model_input"] = df["subset"] + ": " + df["input"]
    df["model_output"] = df["output"]

    return df.reset_index(drop=True)


def clean_test(df):
    df = df.copy()

    df = df.dropna(subset=["input", "subset"])

    df["input"] = df["input"].astype(str).str.strip()
    df["subset"] = df["subset"].astype(str).str.strip()

    df["model_input"] = df["subset"] + ": " + df["input"]

    return df.reset_index(drop=True)


train_df = clean_train_val(train_df)
val_df = clean_train_val(val_df)
test_df = clean_test(test_df)

print("Clean train:", train_df.shape)
print("Clean val:", val_df.shape)
print("Clean test:", test_df.shape)

print("\nExample input:")
print(train_df["model_input"].iloc[0])

print("\nExample output:")
print(train_df["model_output"].iloc[0][:300])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["model_input"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
        text_target=examples["model_output"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


train_ds = Dataset.from_pandas(
    train_df[["model_input", "model_output"]],
    preserve_index=False
)

val_ds = Dataset.from_pandas(
    val_df[["model_input", "model_output"]],
    preserve_index=False
)

train_ds = train_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=train_ds.column_names,
)

val_ds = val_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=val_ds.column_names,
)

print("Tokenized train:", len(train_ds))
print("Tokenized val:", len(val_ds))

In [ ]:
!pip install -q safetensors

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    use_safetensors=True
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

In [ ]:
bad_words_ids = []

for i in range(100):
    token = f"<extra_id_{i}>"
    token_id = tokenizer.convert_tokens_to_ids(token)

    if token_id is not None and token_id != tokenizer.unk_token_id:
        bad_words_ids.append([token_id])

print("Bad token IDs blocked:", len(bad_words_ids))

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="steps",
    eval_steps=2000,

    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=1,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    warmup_steps=500,

    fp16=False,  # safer on P100 for this task
    gradient_checkpointing=True,
    max_grad_norm=1.0,

    logging_steps=50,
    report_to="none",
    dataloader_num_workers=0,
    remove_unused_columns=True,
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
)

trainer.train()

In [ ]:
os.makedirs(BEST_DIR, exist_ok=True)

trainer.save_model(BEST_DIR)
tokenizer.save_pretrained(BEST_DIR)

print("Best model saved to:", BEST_DIR)

In [ ]:
rouge = evaluate.load("rouge")

gc.collect()
torch.cuda.empty_cache()

model = AutoModelForSeq2SeqLM.from_pretrained(
    BEST_DIR,
    use_safetensors=True
).to(device)

model.eval()

N_VAL_EVAL = 500
small_val_df = val_df.iloc[:N_VAL_EVAL].copy()

decoded_preds = []
decoded_refs = small_val_df["model_output"].tolist()

batch_size = 4

for i in tqdm(range(0, len(small_val_df), batch_size), desc="Validation generation"):
    batch_texts = small_val_df["model_input"].iloc[i:i+batch_size].tolist()

    inputs = tokenizer(
        batch_texts,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(device)

    with torch.inference_mode():
        outputs = model.generate(
                **inputs,
                max_length=MAX_TARGET_LENGTH,
                num_beams=4,
                do_sample=False,
                early_stopping=True,
                no_repeat_ngram_size=3,
                repetition_penalty=1.3,
                length_penalty=1.0,
            )

    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    decoded_preds.extend(preds)

    del inputs, outputs
    torch.cuda.empty_cache()

rouge_scores = rouge.compute(
    predictions=decoded_preds,
    references=decoded_refs,
    use_stemmer=False,
    rouge_types=["rouge1", "rougeL"],
)

print(f"Validation ROUGE-1: {rouge_scores['rouge1']:.4f}")
print(f"Validation ROUGE-L: {rouge_scores['rougeL']:.4f}")

In [ ]:
for i in range(10):
    print("=" * 80)
    print("INPUT:")
    print(small_val_df["model_input"].iloc[i])

    print("\nREFERENCE:")
    print(decoded_refs[i])

    print("\nPREDICTION:")
    print(decoded_preds[i])

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForSeq2SeqLM.from_pretrained(BEST_DIR).to(device)
model.eval()

test_inputs = test_df["model_input"].astype(str).tolist()
predictions = []

batch_size = 4

for i in tqdm(range(0, len(test_inputs), batch_size), desc="Generating test predictions"):
    batch = test_inputs[i:i+batch_size]

    inputs = tokenizer(
        batch,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(device)

    with torch.inference_mode():
        outputs = model.generate(
                **inputs,
                max_length=MAX_TARGET_LENGTH,
                num_beams=4,
                do_sample=False,
                early_stopping=True,
                no_repeat_ngram_size=3,
                repetition_penalty=1.3,
                length_penalty=1.0,
            )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions.extend(decoded)

    del inputs, outputs
    torch.cuda.empty_cache()

print("Generated:", len(predictions))
print("Expected:", len(test_df))
print("Sample prediction:", predictions[0][:300])

In [ ]:
# Check empty predictions
submission = pd.DataFrame({
    "ID": test_df["ID"],
    "TargetRLF1": predictions,
    "TargetR1F1": predictions,
    "TargetLLM": predictions,
})

# Clean predictions
for col in ["TargetRLF1", "TargetR1F1", "TargetLLM"]:
    submission[col] = submission[col].astype(str).str.strip()
    submission[col] = submission[col].replace(["", "nan", "None"], np.nan)

# Find empty rows
empty_rows = submission[
    submission[["TargetRLF1", "TargetR1F1", "TargetLLM"]].isna().any(axis=1)
]

print("Empty rows:", len(empty_rows))
display(empty_rows.head())

In [ ]:
fallback_answer = "Please consult a qualified health worker for accurate health guidance."

for col in ["TargetRLF1", "TargetR1F1", "TargetLLM"]:
    submission[col] = submission[col].fillna(fallback_answer)

submission.to_csv("submission.csv", index=False)

print(submission.shape)
print(submission.isna().sum())
print("Saved fixed submission.csv")

In [ ]:
submission[["TargetRLF1", "TargetR1F1", "TargetLLM"]].apply(lambda x: x.str.strip().eq("").sum())

In [ ]:
assert len(predictions) == len(test_df), "Prediction count does not match test rows."

submission = pd.DataFrame({
    "ID": test_df["ID"],
    "TargetRLF1": predictions,
    "TargetR1F1": predictions,
    "TargetLLM": predictions,
})

submission.to_csv("submission.csv", index=False)

print(submission.head())
print("Submission shape:", submission.shape)
print("Saved: submission.csv")

In [ ]:
import os
print(os.listdir("/kaggle/working"))

In [ ]:
from IPython.display import FileLink

FileLink("submission.csv")